# Examen 7: Aprendizaje por Refuerzo

## Evasion de Obstaculos mediante Seleccion de Acciones con Intervalo de Confianza

**Chelsea Melany Espinoza Cava**

**Ing. de Sistemas**

**Segundo Parcial**

El Entorno: Mi entorno consiste en una carretera dividida en 3 carriles discretos (carril0: Izquierda, carril1: Centro, carril2: Derecha). A cada paso que se da el agente avanza un cuadrito hacia adelante y debe decidir en cuál de los 3 carriles quedarse para evitar los obstáculos de la siguiente fila. En cada paso de tiempo se genera un patrón de obstáculos aleatorios en la siguiente fila inmediata. Para asegurar que el agente siempre pueda esquivarlos y sobrevivir, el entorno nunca bloqueará los 3 carriles al mismo tiempo ,como máximo habrá 2 obstáculos por fila para que el agente pueda moverse y no perder.


Los Estados: El estado está determinado por la configuración de obstáculos que se encuentra en la fila inmediatamente siguiente. Representado como una lista binaria de tamaño 3, por ejemplo, (1, 0, 1) indica un obstaculo en el carril izquierdo, y derecho, quedanco el del centro libre, con esta lógica, existen exactamente 2^3 - 1 = 7 estados posibles excluyendo el caso en el que en todos los carriles enten bloqueados -> (1, 1, 1).

Las Acciones: Corresponden al carril que el agente selecciona para el siguiente paso: 0 (Izquierda), 1 (Centro) o 2 (Derecha).


La Recompensa: Se le da un +1 si el agente se pone en un carril libre de obstáculos ya que seria un buen movimiento y estaria a salvo. Pero seria de -10 si escoje un carril en que tenga obstaculo y choca contra ese obstaculo


La funcion de valor: Una matriz de dimensiones 8 x 3 que almacena las estimaciones del valor de cada acción en cada estado posible. Los valores se actualizan dinamicamengte en base a las recompensas recibidas mediante el promedio muestral histórico.

**Exploración y Explotación: Intervalo de Confianza Superior (UCB)**

Para balancear la exploración y explotación de manera óptima, el agente aplicará la estrategia de Selección de Acciones con Intervalo de Confianza (UCB).

Usaremo UCB y no e-greedy por que el e-greedy explora d eforma ciega seleccionanado acciones aleatorias uniformemente sin pensar en que si es bueno o no, asi que UCB mejora esto ya que nos da optimismo frente a la incertidumbre


La política de selección de acción en el estado $s$ en el instante $t$ se define como:

$$
    A_t = \underset{a}{\arg\max} \, \left[ Q_t(a) + c \sqrt{\frac{\text{ln} \, t}{N_t(a)}} \right]
$$

donde $\text{ln} \, t$ es el logaritmo natural de $t$, $N_t(a)$ es el número de veces que se has efectuado la acción $a$ y $c > 0$ es una constante que controla el ratio de exploración.

Donde:
*   $Q_t(a)$ es el valor estimado de tomar la acción $a$ en el estado $s$.
*   $N_t(a)$ es el número de veces que se ha seleccionado la acción $a$ en el estado $s$.
*   $c$ es una constante de exploración ($c > 0$).



Explicacion
El termino de la raiz cuadrada representa la incertidumbre o la varianza en la estimación del valor de la acción. 
* Si una acción a ha sido seleccionada pocas veces, el denominador N_t(a) sera muy pequeño, lo que aumentara significativamente el termino de incertidumbre. Esto eleva el límite superior de confianza de la acción, haciendo que el agente tiende a seleccionarla para recopilar informacion loq ue estaroamos haciendo exploración.
* Cada vez que se elige esa acción, N_t(a) aumenta, reduciendo el termino de incertidumbre. Con el tiempo, a medida que N_t(a) crece y el conocimiento es mucho mas preciso, el valor UCB converge hacia la estimación real Q_t(a), y el agente se decantara por explotar las acciones con mayor valor esperado.
* El logaritmo natural ln(t) garantiza que el termino de exploración crezca muy lentamente con el tiempo total, lo que asegura que la exploracion disminuya gradualmente a favor de la explotación.
* Si N_t(a) = 0, definimos el valor de UCB como infinito para obligar al agente a probar cada acción al menos una vez en cada estado.

Creamos en entorno de simulación de carretera de 3 carriles con obstaculos aleatorios.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ponemos desde un incio lo aleatorio para asegurar resultados que sean aleatrorios
np.random.seed(42)

class ObstaculoEntorno:

    def __init__(self):
        self.n_carril = 3
        self.state = None
        self.reset()
        
    def reset(self):
        #Generamr configuracion de obstaculos inicial
        self.state = self._generar_obstaculos()
        return self._get_state_idx(self.state)
        
    def step(self, action):
        #Determinar si hay algun tipo de choque en el carril elegido por el agente (accion)
        colision = (self.state[action] == 1)
        
        if colision:
            reward = -10.0 #Si es asi restamos
        else:
            reward = 1.0 #Y si no suamamos
            
        # Genera nuevos obstaculos para el siguiente paso de tiempo
        self.state = self._generar_obstaculos()
        siguiente_estado = self._get_state_idx(self.state)
        
        return siguiente_estado, reward, colision
        
    def _generar_obstaculos(self):
        # Genera obstaculos aleatorios y aqui aseguramos que hayan solo 2 carriles con obstaculos
        while True:
            obs = [np.random.choice([0, 1]), np.random.choice([0, 1]), np.random.choice([0, 1])]
            if sum(obs) < self.n_carril:  # Al menos un carril libre
                return obs
                
    def _get_state_idx(self, state):
        # Convierte el estado binario [obs0, obs1, obs2] a un indice decimal (0 a 7)
        return state[0] * 4 + state[1] * 2 + state[2]

In [2]:
class UCBAgente:

    def __init__(self, n_states=8, n_accion=3):
        self.n_states = n_states
        self.n_accion = n_accion
        self.Q = np.zeros((n_states, n_accion))  # Valores Q
        self.N = np.zeros((n_states, n_accion))  # Contadores de visitas por acción
        
    def select_action(self, state, c=1.5):
        # Si hay acciones no visitadas en este estado, seleccionarlas para explorarlas primero
        no_visto = [a for a in range(self.n_accion) if self.N[state, a] == 0]
        if no_visto:
            return np.random.choice(no_visto)
            
        # Calcular valores UCB para cada acción en el estado actual
        total_visits = sum(self.N[state, :])
        ucb_values = np.zeros(self.n_accion)
        
        for a in range(self.n_accion):
            # Formula de UCB
            ucb_values[a] = self.Q[state, a] + c * np.sqrt(np.log(total_visits) / self.N[state, a])
            
        # Elegir la accion que maximiza el valor UCB (desempate aleatorio si hay empates)
        max_val = np.max(ucb_values)
        mejor_accion = np.flatnonzero(ucb_values == max_val)
        return np.random.choice(mejor_accion)
        
    def update(self, state, action, reward):
        # Incrementar el numero de visitas de la accion seleccionada en este estado
        self.N[state, action] += 1
        # Actualizacion de valor Q basada en el promedio muestral incremental
        self.Q[state, action] += (reward - self.Q[state, action]) / self.N[state, action]

Simularemos 

In [3]:
# Parametros de la simulacion
pasos = 1000
c_exploracion = 0.5  # Constante de exploracion

# Inicializacion del entorno y agente
env = ObstaculoEntorno()
agent = UCBAgente()

rewards = []
colisions = []
estados_visitados = []
acciones_realizadas = [] #acciones

state = env.reset()

# Bucle principal de entrenamiento
for step in range(pasos):
    # Seleccionar acción con UCB
    action = agent.select_action(state, c=c_exploracion)
    
    # Ejecutar la accion en el entorno
    next_state, reward, colision = env.step(action)
    
    # Actualizar la Q-Table del agente
    agent.update(state, action, reward)
    
    # Registrar metricas
    rewards.append(reward)
    colisions.append(1 if colision else 0)
    estados_visitados.append(state)
    acciones_realizadas.append(action)
    
    state = next_state

# Calcular metricas para visualizacion
window = 100
moving_avg_reward = np.convolve(rewards, np.ones(window)/window, mode='valid')
cumulative_colisions = np.cumsum(colisions)

print('Simulación completada con éxito.')
print(f'Colisiones totales en {pasos} pasos: {sum(colisions)}')
print(f'Porcentaje de colisiones en los primeros 100 pasos: {sum(colisions[:100])}%')
print(f'Porcentaje de colisiones en los últimos 100 pasos: {sum(colisions[-100:])}%')

Simulación completada con éxito.
Colisiones totales en 1000 pasos: 9
Porcentaje de colisiones en los primeros 100 pasos: 9%
Porcentaje de colisiones en los últimos 100 pasos: 0%


In [4]:
# Diccionario de descripción de estados
states_desc = {
    0: 'No hay obstáculos [0, 0, 0]',
    1: 'Obstáculo carril Derecho [0, 0, 1]',
    2: 'Obstáculo carril Centro [0, 1, 0]',
    3: 'Obstáculo Centro y Derecho [0, 1, 1]',
    4: 'Obstáculo carril Izquierdo [1, 0, 0]',
    5: 'Obstáculo Izquierdo y Derecho [1, 0, 1]',
    6: 'Obstáculo Izquierdo y Centro [1, 1, 0]'
}

q_data = []
for s in sorted(states_desc.keys()):
    q_data.append({
        'Configuración de Obstáculos (Estado)': states_desc[s],
        'Q(s, Izquierda)': agent.Q[s, 0],
        'Q(s, Centro)': agent.Q[s, 1],
        'Q(s, Derecha)': agent.Q[s, 2],
        'Visitas Totales N(s)': sum(agent.N[s, :])
    })

#visualización de tabla
df_q = pd.DataFrame(q_data)
df_q

,Configuración de Obstáculos (Estado),"Q(s, Izquierda)","Q(s, Centro)","Q(s, Derecha)",Visitas Totales N(s)
0,"No hay obstáculos [0, 0, 0]",1.0,1.0,1.0,141.0
1,"Obstáculo carril Derecho [0, 0, 1]",1.0,1.0,-10.0,133.0
2,"Obstáculo carril Centro [0, 1, 0]",1.0,-10.0,1.0,148.0
3,"Obstáculo Centro y Derecho [0, 1, 1]",1.0,-10.0,-10.0,134.0
4,"Obstáculo carril Izquierdo [1, 0, 0]",-10.0,1.0,1.0,153.0
5,"Obstáculo Izquierdo y Derecho [1, 0, 1]",-10.0,1.0,-10.0,149.0
6,"Obstáculo Izquierdo y Centro [1, 1, 0]",-10.0,-10.0,1.0,142.0


Análisis e Interpretacion Resultados


Al observar la tabla final de valores Q_t(a), lo siguiente:


Valores de la Q-Table: En la tabla aprendida, se observa con claridad que el agente asocia valores de Q_t(a) significativamente menores (cercanos a -10 o negativos) a aquellas acciones que conducen a una colision en dicho estado, mientras que las acciones seguras reciben valores cercanos a +1 (o ligeramente mayores/menores dependiendo de la historia). 

Ejemplo (Estado 5: Obstáculo Izquierdo y Derecho (1, 0, 1): Los valores en los carriles Izquierdo (0) y Derecho (2) son negativos, mientras que el carril Centro (1) tiene un valor positivo alto. El agente elegirá siempre el centro, que es la única opción libre.
Ejemplo (Estado 0: Sin obstáculos (0, 0, 0): Todas las acciones tienen valores positivos y similares, indicando que el agente puede transitar por cualquiera de los carriles libremente.

.